In [3]:
# !pip install tensorflow pillow mathplotlib

In [4]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras import layers, models
import pathlib

# Dataset path
data_dir = pathlib.Path(r"C:\Users\turzo\Documents\Neural\flower_photos")

# Load data
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training",
    seed=123, image_size=(224, 224), batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation",
    seed=123, image_size=(224, 224), batch_size=32
)

# Pretrained ResNet-50
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

# Model
model = models.Sequential([
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(5, activation="softmax")
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Train classifier head
model.fit(train_ds, validation_data=val_ds, epochs=3)

# Fine-tune last few layers
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(train_ds, validation_data=val_ds, epochs=2)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - accuracy: 0.7888 - loss: 0.6001 - val_accuracy: 0.8815 - val_loss: 0.3308
Epoch 2/3
61/92 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - accuracy: 0.9045 - loss: 0.2814

KeyboardInterrupt: 